## test concordance

In [2]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install tabulate


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
from IPython.display import display, Markdown

# ----------------------------
# SETTINGS
# ----------------------------
file_path = "~/Desktop/Github/policyclaims/data/gold_standard_30_march.xlsx"
sheet_name = "in"

reference_cols = [
    "llm_policy_claim",
    ]

compare_cols = [
    "agreed_gold_standard",
    "agreed_gold_standard_with_exclusions",
    "DB review",
    "EC review",
    "MW review",
    "db re-review",
    "EC re-review",
]

# ----------------------------
# HELPER: recode binary values
# ----------------------------
na_strings = {"n/a", "na", "nan", "#n/a", "none", "", "-", "."}

def recode_binary(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    if s in na_strings:
        return np.nan

    yes_vals = {"1", "yes", "y", "true"}
    no_vals = {"0", "no", "n", "false"}

    if s in yes_vals:
        return 1
    if s in no_vals:
        return 0

    try:
        f = float(s)
        if f == 1:
            return 1
        if f == 0:
            return 0
    except:
        pass

    return np.nan

# ----------------------------
# LOAD DATA
# ----------------------------
df = pd.read_excel(file_path, sheet_name=sheet_name)

all_cols = reference_cols + compare_cols

missing_cols = []
for col in all_cols:
    if col not in df.columns:
        missing_cols.append(col)
    else:
        df[col + "_bin"] = df[col].apply(recode_binary)

# ----------------------------
# FUNCTION
# ----------------------------
def compare_to_reference(df, reference_col, other_col):
    ref_bin = reference_col + "_bin"
    oth_bin = other_col + "_bin"

    # Only rows where BOTH values are non-missing (0 or 1)
    sub = df[[ref_bin, oth_bin]].dropna()
    sub = sub[sub[ref_bin].isin([0, 1]) & sub[oth_bin].isin([0, 1])]

    if len(sub) == 0:
        return {
            "comparison": f"{reference_col} vs {other_col}",
            "n": 0,
            "agreement_pct": np.nan,
            "kappa": np.nan
        }

    agreement = (sub[ref_bin] == sub[oth_bin]).mean() * 100
    kappa = cohen_kappa_score(sub[ref_bin], sub[oth_bin])

    return {
        "comparison": f"{reference_col} vs {other_col}",
        "n": len(sub),
        "agreement_pct": round(agreement, 1),
        "kappa": round(kappa, 3)
    }

# ----------------------------
# RUN COMPARISONS
# ----------------------------
md = ""

if missing_cols:
    md += "**Missing columns:**\n"
    for col in missing_cols:
        md += f"- `{col}`\n"
    md += "\n"

for ref_col in reference_cols:
    if ref_col in missing_cols:
        continue

    results = []
    for col in compare_cols:
        if col not in missing_cols and col in df.columns:
            results.append(compare_to_reference(df, ref_col, col))

    md += f"## Concordance with `{ref_col}`\n\n"
    if results:
        results_df = pd.DataFrame(results)
        md += results_df.to_markdown(index=False)
    else:
        md += "_No valid comparisons available._"
    md += "\n\n"

display(Markdown(md))


## Concordance with `llm_policy_claim`

| comparison                                               |   n |   agreement_pct |   kappa |
|:---------------------------------------------------------|----:|----------------:|--------:|
| llm_policy_claim vs agreed_gold_standard                 | 204 |            92.6 |   0.803 |
| llm_policy_claim vs agreed_gold_standard_with_exclusions | 197 |            92.9 |   0.805 |
| llm_policy_claim vs DB review                            | 204 |            91.2 |   0.765 |
| llm_policy_claim vs EC review                            |  94 |            89.4 |   0.705 |
| llm_policy_claim vs MW review                            | 104 |            95.2 |   0.882 |
| llm_policy_claim vs db re-review                         |   5 |            80   |   0.545 |
| llm_policy_claim vs EC re-review                         | 204 |            92.6 |   0.803 |



In [5]:
# ----------------------------
# YES/NO TABULATIONS + SENSITIVITY / SPECIFICITY
# llm_policy_claim (rows) vs gold standard (cols/x = human truth)
# ----------------------------

gold_cols = ["agreed_gold_standard", "agreed_gold_standard_with_exclusions"]

label_map = {0: "No", 1: "Yes"}
cat_order = ["No", "Yes", "Total"]

for gold_col in gold_cols:
    if gold_col not in df.columns:
        print(f"Column not found: {gold_col}")
        continue

    ref_bin = "llm_policy_claim_bin"
    oth_bin = gold_col + "_bin"

    sub = df[[ref_bin, oth_bin]].dropna()
    sub = sub[sub[ref_bin].isin([0, 1]) & sub[oth_bin].isin([0, 1])].copy()
    sub[ref_bin] = sub[ref_bin].map(label_map)
    sub[oth_bin] = sub[oth_bin].map(label_map)

    counts = pd.crosstab(
        sub[ref_bin],
        sub[oth_bin],
        rownames=["llm_policy_claim"],
        colnames=[gold_col],
        margins=True,
        margins_name="Total"
    ).reindex(index=cat_order, columns=cat_order, fill_value=0)

    col_pct = pd.crosstab(
        sub[ref_bin],
        sub[oth_bin],
        rownames=["llm_policy_claim"],
        colnames=[gold_col],
        margins=True,
        margins_name="Total",
        normalize="columns"
    ).reindex(index=cat_order, columns=cat_order, fill_value=0).mul(100).round(1)

    combined = counts.astype(str) + " (" + col_pct.astype(str) + "%)"

    display(Markdown(f"### `llm_policy_claim` vs `{gold_col}` — n={len(sub)}"))
    display(Markdown("_Rows = LLM prediction · Columns = human truth · Cell = count (% within gold standard column)_"))
    display(combined)

    TP = counts.loc["Yes", "Yes"]
    FN = counts.loc["No",  "Yes"]
    TN = counts.loc["No",  "No"]
    FP = counts.loc["Yes", "No"]

    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    specificity = TN / (TN + FP) if (TN + FP) > 0 else np.nan
    ppv         = TP / (TP + FP) if (TP + FP) > 0 else np.nan
    npv         = TN / (TN + FN) if (TN + FN) > 0 else np.nan

    def fmt(v):
        return f"{v*100:.1f}%" if not np.isnan(v) else "N/A"

    metrics = pd.DataFrame({
        "Metric":   ["Sensitivity", "Specificity", "PPV", "NPV"],
        "Formula":  ["TP / (TP+FN)", "TN / (TN+FP)", "TP / (TP+FP)", "TN / (TN+FN)"],
        "Value":    [fmt(sensitivity), fmt(specificity), fmt(ppv), fmt(npv)],
    })

    display(Markdown(
        f"\n| | |\n|---|---|\n"
        f"| **TP** | {TP} | **FP** | {FP} |\n"
        f"| **FN** | {FN} | **TN** | {TN} |"
    ))
    display(metrics.set_index("Metric"))
    display(Markdown("---"))

### `llm_policy_claim` vs `agreed_gold_standard` — n=204

_Rows = LLM prediction · Columns = human truth · Cell = count (% within gold standard column)_

agreed_gold_standard,No,Yes,Total
llm_policy_claim,,,
No,146 (91.8%),2 (4.4%),148 (72.5%)
Yes,13 (8.2%),43 (95.6%),56 (27.5%)
Total,159 (0.0%),45 (0.0%),204 (0.0%)



| | |
|---|---|
| **TP** | 43 | **FP** | 13 |
| **FN** | 2 | **TN** | 146 |

,Formula,Value
Metric,,
Sensitivity,TP / (TP+FN),95.6%
Specificity,TN / (TN+FP),91.8%
PPV,TP / (TP+FP),76.8%
NPV,TN / (TN+FN),98.6%


...

_Rows = LLM prediction · Columns = human truth · Cell = count (% within gold standard column)_

agreed_gold_standard_with_exclusions,No,Yes,Total
llm_policy_claim,,,
No,143 (92.3%),2 (4.8%),145 (73.6%)
Yes,12 (7.7%),40 (95.2%),52 (26.4%)
Total,155 (0.0%),42 (0.0%),197 (0.0%)



| | |
|---|---|
| **TP** | 40 | **FP** | 12 |
| **FN** | 2 | **TN** | 143 |

,Formula,Value
Metric,,
Sensitivity,TP / (TP+FN),95.2%
Specificity,TN / (TN+FP),92.3%
PPV,TP / (TP+FP),76.9%
NPV,TN / (TN+FN),98.6%


---

**Revised Code**

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score
from IPython.display import display, Markdown

# =========================================================
# SETTINGS
# =========================================================
file_path = Path("~/Desktop/Github/policyclaims/data/gold_standard_30_march.xlsx").expanduser()
sheet_name = "in"

gold_standard_cols = [
    "agreed_gold_standard",
    "agreed_gold_standard_with_exclusions",
]

automated_cols = [
    "llm_policy_claim",
]

reviewer_cols = [
    "DB review",
    "EC review",
    "MW review",
]

rereview_pairs = [
    ("DB review", "db re-review"),
    ("EC review", "EC re-review"),
]

na_strings = {"n/a", "na", "nan", "#n/a", "none", "", "-", ".", "missing"}
yes_vals = {"1", "yes", "y", "true"}
no_vals = {"0", "no", "n", "false"}

# =========================================================
# HELPER FUNCTIONS
# =========================================================
def recode_binary(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    if s in na_strings:
        return np.nan
    if s in yes_vals:
        return 1
    if s in no_vals:
        return 0

    try:
        f = float(s)
        if f == 1:
            return 1
        if f == 0:
            return 0
    except ValueError:
        pass

    return np.nan


def classification_metrics(df, ref_col, pred_col):
    ref_bin = ref_col + "_bin"
    pred_bin = pred_col + "_bin"

    sub = df[[ref_bin, pred_bin]].dropna().copy()
    sub = sub[sub[ref_bin].isin([0, 1]) & sub[pred_bin].isin([0, 1])].copy()

    if len(sub) == 0:
        return {
            "reference": ref_col,
            "comparison": pred_col,
            "n": 0,
            "agreement_pct": np.nan,
            "kappa": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "ppv": np.nan,
            "npv": np.nan,
        }

    sub[ref_bin] = sub[ref_bin].astype(int)
    sub[pred_bin] = sub[pred_bin].astype(int)

    tp = ((sub[ref_bin] == 1) & (sub[pred_bin] == 1)).sum()
    tn = ((sub[ref_bin] == 0) & (sub[pred_bin] == 0)).sum()
    fp = ((sub[ref_bin] == 0) & (sub[pred_bin] == 1)).sum()
    fn = ((sub[ref_bin] == 1) & (sub[pred_bin] == 0)).sum()

    agreement = (sub[ref_bin] == sub[pred_bin]).mean() * 100
    kappa = cohen_kappa_score(sub[ref_bin], sub[pred_bin])

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

    return {
        "reference": ref_col,
        "comparison": pred_col,
        "n": len(sub),
        "agreement_pct": round(agreement, 1),
        "kappa": round(kappa, 3),
        "sensitivity": round(sensitivity, 3) if pd.notna(sensitivity) else np.nan,
        "specificity": round(specificity, 3) if pd.notna(specificity) else np.nan,
        "ppv": round(ppv, 3) if pd.notna(ppv) else np.nan,
        "npv": round(npv, 3) if pd.notna(npv) else np.nan,
    }


def agreement_only_metrics(df, col1, col2):
    c1 = col1 + "_bin"
    c2 = col2 + "_bin"

    sub = df[[c1, c2]].dropna().copy()
    sub = sub[sub[c1].isin([0, 1]) & sub[c2].isin([0, 1])].copy()

    if len(sub) == 0:
        return {
            "comparison": f"{col1} vs {col2}",
            "n": 0,
            "agreement_pct": np.nan,
            "kappa": np.nan,
        }

    sub[c1] = sub[c1].astype(int)
    sub[c2] = sub[c2].astype(int)

    agreement = (sub[c1] == sub[c2]).mean() * 100
    kappa = cohen_kappa_score(sub[c1], sub[c2])

    return {
        "comparison": f"{col1} vs {col2}",
        "n": len(sub),
        "agreement_pct": round(agreement, 1),
        "kappa": round(kappa, 3),
    }

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_excel(file_path, sheet_name=sheet_name)
df.columns = df.columns.str.strip()

all_cols = list(dict.fromkeys(
    gold_standard_cols + automated_cols + reviewer_cols + [x for pair in rereview_pairs for x in pair]
))

missing_cols = []
for col in all_cols:
    if col not in df.columns:
        missing_cols.append(col)
    else:
        df[col + "_bin"] = df[col].apply(recode_binary)

if missing_cols:
    display(Markdown("## Missing columns"))
    display(pd.DataFrame({"missing_column": missing_cols}))

display(Markdown(
    "All pairwise comparisons below use complete cases for the two columns being compared."
))

# =========================================================
# AUTOMATED METHODS VS GOLD STANDARD
# =========================================================
display(Markdown("## Automated methods versus gold standard"))

auto_results = []
for gs in gold_standard_cols:
    if gs in missing_cols:
        continue
    for auto in automated_cols:
        if auto in missing_cols:
            continue
        auto_results.append(classification_metrics(df, gs, auto))

if auto_results:
    auto_results_df = pd.DataFrame(auto_results)
    display(auto_results_df)
else:
    display(Markdown("_No valid automated versus gold standard comparisons available._"))

# =========================================================
# REVIEWERS VS GOLD STANDARD
# =========================================================
display(Markdown("## Individual reviewers versus gold standard"))

reviewer_results = []
for gs in gold_standard_cols:
    if gs in missing_cols:
        continue
    for reviewer in reviewer_cols:
        if reviewer in missing_cols:
            continue
        reviewer_results.append(classification_metrics(df, gs, reviewer))

if reviewer_results:
    reviewer_results_df = pd.DataFrame(reviewer_results)
    display(reviewer_results_df)
else:
    display(Markdown("_No valid reviewer versus gold standard comparisons available._"))


All pairwise comparisons below use complete cases for the two columns being compared.

## Automated methods versus gold standard

,reference,comparison,n,agreement_pct,kappa,sensitivity,specificity,ppv,npv
0,agreed_gold_standard,llm_policy_claim,204,92.6,0.803,0.956,0.918,0.768,0.986
1,agreed_gold_standard_with_exclusions,llm_policy_claim,197,92.9,0.805,0.952,0.923,0.769,0.986


## Individual reviewers versus gold standard

,reference,comparison,n,agreement_pct,kappa,sensitivity,specificity,ppv,npv
0,agreed_gold_standard,DB review,204,96.6,0.901,0.933,0.975,0.913,0.981
1,agreed_gold_standard,EC review,94,97.9,0.934,0.947,0.987,0.947,0.987
2,agreed_gold_standard,MW review,104,96.2,0.900,1.000,0.949,0.862,1.000
3,agreed_gold_standard_with_exclusions,DB review,197,96.4,0.895,0.929,0.974,0.907,0.981
4,agreed_gold_standard_with_exclusions,EC review,92,97.8,0.934,0.947,0.986,0.947,0.986
5,agreed_gold_standard_with_exclusions,MW review,101,96.0,0.894,1.000,0.949,0.852,1.000
